# Extract and Consolidate Tables from Word Document

This notebook extracts tables from the Word document and consolidates them into a single CSV file.


In [ ]:
!uv pip install python-docx


Using Python 3.11.14 environment at: /home/abhishek/Documents/heatpump_ai_2/climate_data_prep/.venv
Resolved 3 packages in 109ms                                         
Installed 1 package in 23ms                                 
 + python-docx==1.2.0


In [ ]:
from docx import Document
import csv
import os
import pandas as pd
from pathlib import Path


In [ ]:
def extract_tables_to_csv(docx_path, output_dir):
    """
    Extract all tables from a .docx file and save each as a CSV.
    
    Args:
        docx_path: Path to the Word document
        output_dir: Directory to save CSV files
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    document = Document(docx_path)

    for table_index, table in enumerate(document.tables, start=1):
        csv_path = os.path.join(output_dir, f"table_{table_index}.csv")

        with open(csv_path, mode="w", newline="", encoding="utf-8") as csv_file:
            writer = csv.writer(csv_file)

            for row in table.rows:
                row_data = []
                for cell in row.cells:
                    # Normalize whitespace and line breaks
                    text = cell.text.replace("\n", " ").strip()
                    row_data.append(text)

                writer.writerow(row_data)

        print(f"Saved: {csv_path}")


In [ ]:
# Configuration
DOCX_FILE = "beg_waermepumpen_pruef_effizienznachweis.docx"
OUTPUT_BASE_DIR = "/mnt/d/heatpump_data/heatpump_table"
OUTPUT_DIR = os.path.join(OUTPUT_BASE_DIR, "tables_csv")
CONSOLIDATED_CSV = os.path.join(OUTPUT_BASE_DIR, "consolidated_table.csv")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Extract tables to separate CSVs
extract_tables_to_csv(DOCX_FILE, OUTPUT_DIR)


Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_1.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_2.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_3.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_4.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_5.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_6.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_7.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_8.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_9.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_10.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_11.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_12.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_13.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_14.csv
Saved: /mnt/d/heatpump_data/heatpump_table/tables_csv/table_15.csv
Save

In [ ]:
# Consolidate all CSV files into a single CSV
csv_files = sorted(Path(OUTPUT_DIR).glob("table_*.csv"))

if not csv_files:
    print("No CSV files found!")
else:
    print(f"Found {len(csv_files)} CSV files to consolidate")
    
    # First pass: find maximum column count and get reference header from a 9-column file
    max_cols = 0
    reference_header = None
    
    # First, find the maximum column count
    for csv_file in csv_files:
        df = pd.read_csv(csv_file, encoding="utf-8")
        num_cols = len(df.columns)
        if num_cols > max_cols:
            max_cols = num_cols
    
    # Then, find a file with 9 columns (or max_cols) to use as reference header
    for csv_file in csv_files:
        df = pd.read_csv(csv_file, encoding="utf-8")
        if len(df.columns) == 9:  # Specifically look for 9-column files
            reference_header = df.columns.tolist()
            print(f"Using reference header from {csv_file.name} (9 columns)")
            break
    
    # Fallback: if no 9-column file found, use max_cols file
    if reference_header is None:
        for csv_file in csv_files:
            df = pd.read_csv(csv_file, encoding="utf-8")
            if len(df.columns) == max_cols:
                reference_header = df.columns.tolist()
                print(f"Using reference header from {csv_file.name} ({max_cols} columns)")
                break
    
    print(f"Maximum columns found: {max_cols}")
    print(f"Reference header: {reference_header}\n")
    
    # Second pass: read and consolidate all CSVs
    dataframes = []
    
    for csv_file in csv_files:
        df = pd.read_csv(csv_file, encoding="utf-8")
        num_cols = len(df.columns)
        original_num_cols = num_cols
        
        # Pad columns if needed - add empty columns until we have max_cols
        # We'll rename all columns at the end, so we don't need to worry about names now
        while len(df.columns) < max_cols:
            # Add a temporary column with a unique name
            df[f'_pad_col_{len(df.columns)}'] = ''
        
        # Now rename all columns to match reference header
        # This handles cases where original column names differ from reference
        df.columns = reference_header
        
        if original_num_cols < max_cols:
            print(f"✓ Padded {csv_file.name}: {original_num_cols} -> {max_cols} columns, {len(df)} rows")
        else:
            print(f"Processed {csv_file.name}: {num_cols} columns, {len(df)} rows")
        
        # Ensure column order matches reference header
        df = df[reference_header]
        
        # Handle header detection - check if first row is a header row
        if len(df) > 0:
            first_row_values = [str(x).strip() for x in df.iloc[0].tolist()]
            header_values = [str(x).strip() for x in reference_header]
            
            # Check if first row matches header (repeated header row)
            # Use multiple methods to detect headers:
            # 1. Exact match
            # 2. Check if first row contains common header keywords
            # 3. Check if most values match (allowing for minor differences)
            
            is_header_row = False
            
            # Method 1: Exact match
            if first_row_values == header_values:
                is_header_row = True
            else:
                # Method 2: Check if first row contains header keywords
                header_keywords = ['Hersteller', 'Typ', 'Kältemittel', 'Niedertemperatur', 'Wärme', 'ETAs', 
                                 'Verfügbarkeit', 'Netzdien', 'EE-Anzeige']
                first_row_text = ' '.join(first_row_values).lower()
                keyword_matches = sum(1 for keyword in header_keywords if keyword.lower() in first_row_text)
                
                # If 3+ header keywords found in first row, likely a header
                if keyword_matches >= 3:
                    is_header_row = True
                else:
                    # Method 3: Check if values match (case-insensitive, allowing for minor differences)
                    matches = sum(1 for i, (val1, val2) in enumerate(zip(first_row_values, header_values)) 
                                if val1.lower() == val2.lower() or val1.lower() in val2.lower() or val2.lower() in val1.lower())
                    # If 70%+ of columns match, likely a header
                    if matches >= len(reference_header) * 0.7:
                        is_header_row = True
            
            if is_header_row:
                # First row is header, skip it
                df = df.iloc[1:].reset_index(drop=True)
                if len(df) > 0:  # Only append if there are data rows left
                    dataframes.append(df)
            else:
                # Regular data rows, append all
                dataframes.append(df)
    
    # Concatenate all dataframes
    if dataframes:
        consolidated_df = pd.concat(dataframes, ignore_index=True)
        
        # Save consolidated CSV
        consolidated_df.to_csv(CONSOLIDATED_CSV, index=False, encoding="utf-8")
        
        print(f"\nConsolidated table saved to: {CONSOLIDATED_CSV}")
        print(f"Total rows: {len(consolidated_df)}")
        print(f"Total columns: {len(consolidated_df.columns)}")
    else:
        print("No tables could be consolidated!")


Found 441 CSV files to consolidate
Using reference header from table_100.csv (9 columns)
Maximum columns found: 9
Reference header: ['Hersteller', 'Typ', 'Niedertemperatur- Anwendung 35 °C', 'Niedertemperatur- Anwendung 35 °C.1', 'Niedertemperatur- Anwendung 55 °C', 'Niedertemperatur- Anwendung 55 °C.1', 'Kältemittel', 'Verfügbarkeit (Siehe Hinweis auf Seite 5)', 'Verfügbarkeit (Siehe Hinweis auf Seite 5).1']

✓ Padded table_10.csv: 8 -> 9 columns, 22 rows
Processed table_100.csv: 9 columns, 32 rows
Processed table_101.csv: 9 columns, 32 rows
Processed table_102.csv: 9 columns, 32 rows
Processed table_103.csv: 9 columns, 31 rows
Processed table_104.csv: 9 columns, 19 rows
Processed table_105.csv: 9 columns, 19 rows
Processed table_106.csv: 9 columns, 19 rows
Processed table_107.csv: 9 columns, 30 rows
Processed table_108.csv: 9 columns, 26 rows
Processed table_109.csv: 9 columns, 31 rows
✓ Padded table_11.csv: 8 -> 9 columns, 32 rows
Processed table_110.csv: 9 columns, 23 rows
Processe

PermissionError: [Errno 13] Permission denied: '/mnt/d/heatpump_data/heatpump_table/consolidated_table.csv'

In [27]:
# Load consolidated CSV and remove category header rows
consolidated_df = pd.read_csv(CONSOLIDATED_CSV, encoding="utf-8")

print(f"Original rows: {len(consolidated_df)}")

# Identify rows that are likely category headers (e.g., "Luft / Wasser", "Abluft / Wasser", etc.)
# These are rows where most columns are empty and the first column contains category names
category_patterns = ['Luft / Wasser', 'Abluft / Wasser', 'Luft / Luft', 'Direktverdampfung / Wasser', 
                    'Erdreich / Wasser', 'Sole / Wasser', 'Wasser / Wasser']

# Drop rows where:
# 1. First column contains a category pattern, OR
# 2. First column is non-empty but most other columns are empty (likely a header row)
rows_to_drop = []

for idx, row in consolidated_df.iterrows():
    first_col = str(row.iloc[0]).strip()
    
    # Check if first column matches a category pattern
    if any(pattern in first_col for pattern in category_patterns):
        rows_to_drop.append(idx)
    # Check if first column is non-empty but most other columns (2-9) are empty
    elif first_col and first_col != 'nan':
        non_empty_count = sum(1 for i in range(1, len(row)) if str(row.iloc[i]).strip() and str(row.iloc[i]).strip() != 'nan')
        if non_empty_count <= 1:  # If only 0-1 other columns have data, likely a header
            rows_to_drop.append(idx)

# Drop the identified rows
consolidated_df_cleaned = consolidated_df.drop(index=rows_to_drop).reset_index(drop=True)

print(f"Rows dropped: {len(rows_to_drop)}")
print(f"Remaining rows: {len(consolidated_df_cleaned)}")

# Save cleaned consolidated CSV
consolidated_df_cleaned.to_csv(CONSOLIDATED_CSV, index=False, encoding="utf-8")
print(f"\nCleaned consolidated table saved to: {CONSOLIDATED_CSV}")


Original rows: 12441
Rows dropped: 442
Remaining rows: 11999

Cleaned consolidated table saved to: /mnt/d/heatpump_data/heatpump_table/consolidated_table.csv


In [28]:
# Display first few rows of consolidated table
consolidated_df.head()


,Hersteller,Typ,Niedertemperatur- Anwendung 35 °C,Niedertemperatur- Anwendung 35 °C.1,Niedertemperatur- Anwendung 55 °C,Niedertemperatur- Anwendung 55 °C.1,Kältemittel,Verfügbarkeit (Siehe Hinweis auf Seite 5),Verfügbarkeit (Siehe Hinweis auf Seite 5).1
0,NaN,ηs (bei 35 °C),ηs (bei 55 °C),NaN,NaN,NaN,NaN,NaN,NaN
1,Wärmequelle Luft,145%,125%,NaN,NaN,NaN,NaN,NaN,NaN
2,Wärmequelle Erdwärme,180%,140%,NaN,NaN,NaN,NaN,NaN,NaN
3,Wärmequelle Wasser,180%,140%,NaN,NaN,NaN,NaN,NaN,NaN
4,"Sonstige Wärmequellen (z.B. Abwärme, Solarwärme)",180%,140%,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Display basic info about the consolidated table
consolidated_df.info()
